# BirdCLEF+ 2026 — PERCH v2 Baseline

This notebook is intended to be **run on Kaggle** (CPU only, 90-min limit).

Reference public notebook: `kaggle.com/code/kailyn2359/birdclef-2026-baseline-with-perch-v2-model`

Expected score: ~0.73 ROC-AUC

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────
import os, time, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# Kaggle competition input path
COMP_DIR = '/kaggle/input/birdclef-2026'
MODEL_DIR = '/kaggle/input/perch-v2'  # Kaggle dataset with PERCH v2 TFLite model

sample_sub = pd.read_csv(f'{COMP_DIR}/sample_submission.csv')
print('Submission shape:', sample_sub.shape)
print('Columns:', sample_sub.columns[:5].tolist(), '...')

In [ ]:
# ── Load PERCH v2 via TFLite ───────────────────────────────────────
# PERCH v2 TFLite model is ~10x faster than the full TF version
import tensorflow as tf

model_path = f'{MODEL_DIR}/model.tflite'  # adjust path as needed

interpreter = tf.lite.Interpreter(model_path=model_path, num_threads=4)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('Input shape:', input_details[0]['shape'])
print('Output shape:', output_details[0]['shape'])

In [ ]:
# ── Audio preprocessing ────────────────────────────────────────────
import librosa

SAMPLE_RATE = 32000
CHUNK_DURATION = 5  # seconds
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_DURATION

def load_and_chunk(path, sr=SAMPLE_RATE):
    """Load audio and split into 5-second chunks."""
    audio, _ = librosa.load(path, sr=sr, mono=True)
    # Pad if shorter than one chunk
    if len(audio) < CHUNK_SAMPLES:
        audio = np.pad(audio, (0, CHUNK_SAMPLES - len(audio)))
    chunks = []
    for start in range(0, len(audio) - CHUNK_SAMPLES + 1, CHUNK_SAMPLES):
        chunks.append(audio[start:start + CHUNK_SAMPLES])
    return chunks

def predict_chunk(audio_chunk):
    """Run PERCH v2 inference on a single 5-second chunk."""
    inp = audio_chunk.reshape(1, -1).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], inp)
    interpreter.invoke()
    logits = interpreter.get_tensor(output_details[0]['index'])[0]
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    return probs

In [ ]:
# ── Build species → column index mapping ──────────────────────────
# sample_submission columns after 'row_id': one per species
species_cols = [c for c in sample_sub.columns if c != 'row_id']
print(f'{len(species_cols)} species columns')

# PERCH v2 output order (from model metadata / class list)
# This mapping aligns PERCH output indices to competition species columns
# Load from model metadata if available; placeholder here:
perch_classes = species_cols  # placeholder — update with actual PERCH class list

In [ ]:
# ── Inference loop ─────────────────────────────────────────────────
soundscape_dir = f'{COMP_DIR}/test_soundscapes'
soundscape_files = sorted(glob.glob(f'{soundscape_dir}/*.ogg'))
print(f'Found {len(soundscape_files)} test soundscapes')

results = []
start_time = time.time()

for i, fpath in enumerate(soundscape_files):
    file_id = os.path.splitext(os.path.basename(fpath))[0]
    chunks = load_and_chunk(fpath)
    
    for chunk_idx, chunk in enumerate(chunks):
        end_sec = (chunk_idx + 1) * CHUNK_DURATION
        row_id = f'{file_id}_{end_sec}'
        probs = predict_chunk(chunk)
        row = {'row_id': row_id}
        row.update(dict(zip(perch_classes, probs)))
        results.append(row)
    
    elapsed = time.time() - start_time
    if (i + 1) % 10 == 0:
        print(f'[{i+1}/{len(soundscape_files)}] {elapsed/60:.1f} min elapsed')

print(f'Total inference time: {(time.time()-start_time)/60:.1f} min')

In [ ]:
# ── Build submission ───────────────────────────────────────────────
pred_df = pd.DataFrame(results)

# Align to sample submission (fill any missing species with 0)
sub = sample_sub[['row_id']].merge(pred_df, on='row_id', how='left')
for col in species_cols:
    if col not in sub.columns:
        sub[col] = 0.0
sub[species_cols] = sub[species_cols].fillna(0.0)

sub = sub[sample_sub.columns]  # ensure column order
print('Submission shape:', sub.shape)
sub.head(3)

In [ ]:
sub.to_csv('submission.csv', index=False)
print('Saved submission.csv')